# Temporal Consistency failure audit — stored results

음의 Δ 18 sequence, 양의 대조군 3개, 결측 사례 2개를 확인한 기록이다.
EXP-005: `docs/reports/experiments/EXP-005-temporal-consistency-failure-audit.md`의 설명과 구분해 읽는다. 모든 관찰은 Codex의 시각 screening이며 독립 human GT가 아니다.
원 추론/학습은 다시 실행하지 않는다. 원본 experiment의 mask/metrics는 보존한다.

재생성: `experiments/reference/rovf/.venv/Scripts/python.exe scripts/audit_temporal_consistency.py`.
새 audit의 시각 판정은 pending으로 시작한다. 아래 `AUDIT_DIR`를 검토하려는 디렉터리로 지정한다.
이 노트북의 기본 경로는 이번에 실제 검토한 audit이다. 다른 환경에서는 데이터·mask·audit artifacts가 필요하다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, uuid
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
ROOT=globals().get('PILOT_ROOT',Path.cwd()).resolve()
while not (ROOT/'pyproject.toml').is_file():
    if ROOT==ROOT.parent: raise RuntimeError('Open from inside repository')
    ROOT=ROOT.parent
AUDIT_DIR=ROOT/'experiments/polarbear/temporal_consistency_identity_test/20261124T120950Z-d2ad62/audits/20261124T121830Z-c8fef7'
metadata=json.loads((AUDIT_DIR/'audit_metadata.json').read_text(encoding='utf-8'))
RUN_DIR=AUDIT_DIR/'notebook_reviews'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:6])
RUN_DIR.mkdir(parents=True,exist_ok=False)
review=pd.read_csv(AUDIT_DIR/'sequence_review.csv')
print('Audit:',metadata['review_status'])
print(review[['clip','delta','visual_motion','visual_mask']].to_string(index=False))
print(pd.read_csv(AUDIT_DIR/'negative_positive_group_medians.csv').to_string(index=False))

## 18개 실패와 대조군의 frame 관찰

각 contact sheet 위 행은 unmasked RGB, 아래 행은 생성 mask overlay이다.
마스크에서 동물 fur를 누락한 지점은 품질 결함의 구체적 예시지만 IoU나 일반적 오류율이 아니다.
Mask=0 한 시점만으로 temporal pure Non-target pair가 됐다고 간주하지 않는다.

In [ ]:
for key in sorted(review['clip']):
    path=AUDIT_DIR/f'{key}_all_frames.png'
    fig,ax=plt.subplots(figsize=(16,4.7))
    ax.imshow(Image.open(path));ax.axis('off');fig.tight_layout();plt.show()

## Mask Quality spot checks

7개 점은 오류가 의심되거나 대비가 되는 위치를 골랐다. 전부 RGB에서 보이는 동물 fur로
Codex가 판단했지만 independent human label은 없다. 5개는 mask=0, 2개는 mask=1이었다.
원본 source의 mask_qc.csv를 pass로 바꾸지 않았다. prompt 위치와 전파 위치 모두 별도 검토해야 한다.

In [ ]:
print(pd.read_csv(AUDIT_DIR/'mask_pixel_probes.csv').to_string(index=False))
fig,ax=plt.subplots(figsize=(8,22));ax.imshow(Image.open(AUDIT_DIR/'mask_pixel_probes.png'))
ax.axis('off');fig.tight_layout();plt.show()

## 원본 영상 확보 판단 및 다음 결정

로컬 ZIP은 JPG 138,363개와 CSV 7개이고, data/ 아래 원본 video file은 확인되지 않았다.
[공식 release](https://zenodo.org/records/7564529)는 하나의 ZIP을 제공한다.
[논문](https://www.mdpi.com/2076-2615/13/5/801)은 개체를 추적하여 crop해 저장하는 과정을 설명한다.
uncropped 공개 다운로드는 Not verified이며 원본 자체가 없다는 결론은 아니다.
PolarBearVidID의 zoo 영상과 camera-trap 도메인을 동일시하지 않는다.

우선 독립 target mask subset을 확보하고, 원본 RGB/카메라 ID/시간/crop transform 및 별도 occluder
label이 준비된 다음 full-background와 occluder 가설을 검증한다. 저자 연락/원본 다운로드는 수행하지 않았다.

In [ ]:
print(json.dumps(json.loads((AUDIT_DIR/'raw_video_availability.json').read_text(encoding='utf-8')),ensure_ascii=False,indent=2))